# RAG Generation Pipeline

**Thesis:** RAG-Driven Natural Language Energy Demand Forecasting  
**Author:** Zoheb Anwar Hussain (Student ID: 1196931)  
**Model:** Llama 3.3 70B via Groq (`llama-3.3-70b-versatile`)

---

### What this notebook does
For each golden query × pipeline combination from Phase 4, retrieves
KB context and generates a natural language answer using Llama 3.3 70B.
The generated answers are compared against Gemini 2.5 Flash reference
answers in Phase 6 (RAGAS evaluation).

### LCEL Chain Pattern
```
retriever → format_docs → prompt → ChatGroq (Llama 3.3 70B) → StrOutputParser
```

### This is where Groq API enters
Phases 1-4 used Gemini (Google AI) and local sentence-transformers.  
This phase uses Groq for the first time — requires `GROQ_API_KEY` in `.env`.

---
## Cell 1 — Imports and Setup

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from config import PATHS, RAG_MODEL_NAME
from config.paths import create_all_directories
from src.utils import setup_logger, log_section
from src.embedding import (
    load_kb_documents,
    get_embeddings_model,
    load_faiss_index,
)
from src.retrieval import (
    DenseRetriever,
    HybridRetriever,
    HierarchicalRetriever,
)
from src.rag import get_rag_llm, generate_rag_answers

logger = setup_logger("rag_generation")
create_all_directories()

logger.info("All imports successful.")
logger.info("RAG model: %s", RAG_MODEL_NAME)

2026-04-29 21:11:24 | INFO     | rag_generation | All imports successful.
2026-04-29 21:11:24 | INFO     | rag_generation | RAG model: llama-3.3-70b-versatile


---
## Cell 2 — Load FAISS Index, Documents, and Golden Dataset

In [2]:
log_section("Loading indexes and datasets", 1, 4)

# Load embedding model + FAISS index
embeddings = get_embeddings_model()
faiss_vs   = load_faiss_index(PATHS["faiss_index"], embeddings)

# Load KB documents (needed by BM25 and hierarchical)
kb_csv_path = PATHS["summaries_csv"] / "combined_master_summaries.csv"
documents   = load_kb_documents(kb_csv_path)

# Load golden dataset
golden_df = pd.read_csv(
    PATHS["golden_dataset"] / "combined_golden_dataset.csv"
)

# Load retrieval results from Phase 4
retrieval_results_df = pd.read_csv(
    PATHS["retrieval_results"] / "retrieval_results.csv"
)

logger.info(
    "Loaded: %d documents, %d golden queries, %d retrieval results.",
    len(documents), len(golden_df), len(retrieval_results_df),
)


  STEP 1/4  [█░░░] 25%
  Loading indexes and datasets



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

2026-04-29 21:13:31 | INFO     | rag_generation | Loaded: 140 documents, 50 golden queries, 93 retrieval results.


---
## Cell 3 — Initialise Retrievers and Groq LLM

In [3]:
log_section("Initialising retrievers and LLM", 2, 4)

K = 5

# Initialise all three retrievers
dense_retriever        = DenseRetriever(faiss_vs, k=K)
hybrid_retriever       = HybridRetriever(faiss_vs, documents, k=K, weights=[0.3, 0.7])
hierarchical_retriever = HierarchicalRetriever(faiss_vs, documents, k=K)

retrievers = {
    "dense":        dense_retriever,
    "hybrid":       hybrid_retriever,
    "hierarchical": hierarchical_retriever,
}

# Initialise Groq LLM (Llama 3.3 70B)
llm = get_rag_llm()

logger.info("All retrievers and LLM initialised.")


  STEP 2/4  [██░░] 50%
  Initialising retrievers and LLM



2026-04-29 21:13:57 | INFO     | rag_generation | All retrievers and LLM initialised.


---
## Cell 4 — Generate RAG Answers

For each query-pipeline combination from Phase 4 retrieval results,
retrieves context and generates an answer via Groq.

**Resume-safe** — already-generated answers are skipped on re-run.

Estimated runtime: ~93 combinations × ~5s per call ≈ **8-10 minutes**.

In [4]:
log_section("Generating RAG answers via Groq", 3, 4)

rag_results_df = generate_rag_answers(
    golden_df=golden_df,
    retrieval_results_df=retrieval_results_df,
    retrievers=retrievers,
    llm=llm,
    output_path=PATHS["rag_results"] / "rag_answers.csv",
)

logger.info("RAG generation complete: %d answers.", len(rag_results_df))


  STEP 3/4  [███░] 75%
  Generating RAG answers via Groq



Generating RAG answers:   0%|          | 0/93 [00:00<?, ?it/s]

2026-04-29 21:22:19 | INFO     | rag_generation | RAG generation complete: 93 answers.


---
## Cell 5 — Quick Quality Check

In [5]:
log_section("Quick quality check", 4, 4)

print("\n" + "=" * 65)
print("  RAG GENERATION — SUMMARY")
print("=" * 65)

print(f"\n  Total RAG answers: {len(rag_results_df)}")

# Answers per pipeline
print("\n  By pipeline:")
print(rag_results_df["pipeline"].value_counts().to_string())

# Error check
errors = rag_results_df[
    rag_results_df["rag_answer"].str.startswith("ERROR:", na=False)
]
if len(errors) > 0:
    print(f"\n  Errors: {len(errors)} answers failed")
    print(errors[["golden_id", "pipeline", "rag_answer"]].to_string())
else:
    print("\n  Errors: None — all answers generated successfully.")

# Answer length stats
rag_results_df["answer_words"] = (
    rag_results_df["rag_answer"].str.split().str.len()
)
print(f"\n  Answer length (words): "
      f"mean={rag_results_df['answer_words'].mean():.0f} | "
      f"min={rag_results_df['answer_words'].min()} | "
      f"max={rag_results_df['answer_words'].max()}")

# Sample answers
print("\n" + "-" * 65)
print("  SAMPLE RAG ANSWERS")
print("-" * 65)
for _, row in rag_results_df.sample(
    n=min(2, len(rag_results_df)), random_state=42
).iterrows():
    print(f"\n  [{row['pipeline']}] golden_id={row['golden_id']}")
    print(f"  Q: {row['user_query'][:80]}...")
    answer = str(row['rag_answer'])[:300]
    ellipsis = '...' if len(str(row['rag_answer'])) > 300 else ''
    print(f"  A: {answer}{ellipsis}")

print("\n" + "=" * 65)


  STEP 4/4  [████] 100%
  Quick quality check


  RAG GENERATION — SUMMARY

  Total RAG answers: 93

  By pipeline:
pipeline
dense           50
hybrid          23
hierarchical    20

  Errors: None — all answers generated successfully.

  Answer length (words): mean=94 | min=63 | max=136

-----------------------------------------------------------------
  SAMPLE RAG ANSWERS
-----------------------------------------------------------------

  [hybrid] golden_id=25
  Q: Which sub-metering channel (kitchen, laundry, or HVAC and water heater) contribu...
  A: The HVAC and water heater sub-metering channel contributes the most to household electricity consumption on average, accounting for 64.6% to 78.4% of the total sub-metered usage across the different time periods. Specifically, the average contribution ranges from 64.6% (week ending December 31, 2006...

  [dense] golden_id=15
  Q: Describe the daily demand behaviour of a specific GEFCom zone and explain how it...
  A: For Zone 11.0 i

---
## Pipeline Complete ✅

RAG answers saved to:
```
outputs/rag_results/rag_answers.csv
```

Columns: `golden_id`, `pipeline`, `user_query`, `retrieved_context`,
`rag_answer`, `reference_answer`, `generated_at`

### Next Phase
Move to `notebooks/06_evaluation.ipynb` to evaluate RAG answers using
RAGAS metrics (faithfulness, answer relevancy, context precision, context recall)
and retrieval metrics (Recall@k, Precision@k, MRR, nDCG).